[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/04_llm_finetuning/prompt_instruction_tuning.ipynb)

# 04 . Prompt and Instruction Tuning

## Concept — Prompt & instruction tuning (revision notes)

**What it is**
- **Instruction tuning (SFT)** continues training a language model on `(prompt, response)` pairs so it follows instructions instead of merely continuing text. The pairs are rendered into one token stream with a **prompt format / chat template**, and the loss is usually computed **only on the response tokens**.
- **Prompt tuning** is the parameter-efficient cousin: the model is frozen and a few continuous "soft prompt" vectors prepended to the input embeddings are learned.

**Why it matters**
- The template is part of the model's interface: train with one format and serve with another and quality collapses. Loss masking, padding and packing decide what the model actually learns and how much compute you waste.

**When to use**
- SFT: to teach a base model a task format, a persona, or tool use. Prompt tuning: when you want a tiny per-task artifact (a few vectors) on top of a frozen, shared model.

**Math & intuition**
- Formats: Alpaca-style `### Instruction / ### Response` text; chat formats (ChatML-style `<|im_start|>role ... <|im_end|>`) built from a list of `{role, content}` messages.
- Loss masking: for tokens `t_1..t_n`, minimise `-sum_i m_i log p(t_i | t_<i)` with `m_i = 1` only on response tokens (labels of masked positions are `-100`). Masking prevents spending capacity on memorising prompts and avoids rewarding the model for reproducing user text.
- Packing: concatenate many short examples (separated by an end token) into fixed-length rows to remove padding waste; use a **block-diagonal causal mask** and reset position ids so examples cannot attend to each other.
- Soft prompts: input becomes `[P_1..P_k ; embed(x)]` with `P` trainable (`k * d_model` parameters).

### 1. Instruction data formats: Alpaca records and chat messages

**What this cell does (plain language):** we generate a small synthetic instruction dataset of three text-editing tasks (`up` = uppercase, `rev` = reverse, `dbl` = double every letter) on random 3-letter words, and store each example in the two dominant record layouts: an Alpaca-style dict (`instruction` / `input` / `output`) and a chat-style list of `{role, content}` messages.

In [1]:
import torch, torch.nn as nn, torch.nn.functional as F, random, math, time, itertools
random.seed(0); torch.manual_seed(0)

TASKS = {"up": lambda w: w.upper(), "rev": lambda w: w[::-1], "dbl": lambda w: "".join(c * 2 for c in w)}
letters = "abcdef"
all_words = ["".join(p) for p in itertools.product(letters, repeat=3)]       # 216 possible words
random.shuffle(all_words)
val_words, train_words = all_words[:30], all_words[30:]                      # held-out words test generalisation

def alpaca_record(task, w):
    return {"instruction": f"{task} the word", "input": w, "output": TASKS[task](w)}
def chat_record(task, w):
    return {"messages": [{"role": "user", "content": f"{task} {w}"}, {"role": "assistant", "content": TASKS[task](w)}]}

print("Alpaca record:", alpaca_record("rev", "abc"))
print("Chat record  :", chat_record("rev", "abc"))
print("words train/val:", len(train_words), len(val_words))

Alpaca record: {'instruction': 'rev the word', 'input': 'abc', 'output': 'cba'}
Chat record  : {'messages': [{'role': 'user', 'content': 'rev abc'}, {'role': 'assistant', 'content': 'cba'}]}
words train/val: 186 30


### 2. The Alpaca prompt template, by hand

**What this cell does (plain language):** we render an Alpaca record into the classic text template with `### Instruction:` / `### Input:` / `### Response:` headers. Records without an input use a shorter variant. The *prompt* part ends right after the `### Response:` header, so the model's job is everything after it.

In [2]:
ALPACA_WITH_INPUT = "Below is an instruction paired with an input. Write a response.\n\n### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n"
ALPACA_NO_INPUT = "Below is an instruction. Write a response.\n\n### Instruction:\n{instruction}\n\n### Response:\n"

def alpaca_prompt(rec):
    tpl = ALPACA_WITH_INPUT if rec.get("input") else ALPACA_NO_INPUT
    return tpl.format(**rec)

rec = alpaca_record("dbl", "cab")
prompt = alpaca_prompt(rec)
full_text = prompt + rec["output"] + "<eos>"
print(full_text)
print("--- prompt chars:", len(prompt), "| response chars:", len(rec["output"]) + len("<eos>"))

Below is an instruction paired with an input. Write a response.

### Instruction:
dbl the word

### Input:
cab

### Response:
ccaabb<eos>
--- prompt chars: 126 | response chars: 11


### 3. A tokenizer with special tokens and a ChatML-style chat template

**What this cell does (plain language):** we build a character-level tokenizer in which `<|im_start|>`, `<|im_end|>`, `<|endoftext|>` and `<pad>` are each a single token (split out with a regex before character encoding), then write `apply_chat_template(messages, add_generation_prompt)` by hand. The generation prompt opens an assistant turn so the model continues from exactly where training responses begin.

In [3]:
import re
SPECIALS = ["<pad>", "<|im_start|>", "<|im_end|>", "<|endoftext|>"]
chars = sorted(set("abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ 0123456789\n.,:!?#"))
itos = SPECIALS + chars; stoi = {t: i for i, t in enumerate(itos)}
PAD, IM_START, IM_END, EOT = 0, 1, 2, 3
split_re = re.compile("(" + "|".join(re.escape(s) for s in SPECIALS) + ")")

def encode(text):
    ids = []
    for piece in split_re.split(text):
        if piece in stoi and piece in SPECIALS: ids.append(stoi[piece])       # whole special token
        else: ids += [stoi[c] for c in piece]
    return ids
def decode(ids, skip_special=False):
    return "".join(itos[i] for i in ids if not (skip_special and i < len(SPECIALS)))

def apply_chat_template(messages, add_generation_prompt=False):
    s = "".join(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>\n" for m in messages)
    return s + ("<|im_start|>assistant\n" if add_generation_prompt else "")

msgs = [{"role": "system", "content": "You edit text."}, {"role": "user", "content": "rev abc"}, {"role": "assistant", "content": "cba"},
        {"role": "user", "content": "up abc"}]
txt = apply_chat_template(msgs, add_generation_prompt=True)
print(txt)
ids = encode(txt)
print("n tokens:", len(ids), "| special ids in stream:", [i for i in ids if i < 4][:8], "...")
print("round trip ok:", decode(ids) == txt)
try:
    from transformers import AutoTokenizer      # real API: tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    print("transformers present: apply_chat_template reads the template stored in tokenizer_config.json (needs a local/downloaded tokenizer)")
except ImportError:
    print("transformers not installed -> pip install transformers (same idea: a Jinja template stored with the tokenizer)")

<|im_start|>system
You edit text.<|im_end|>
<|im_start|>user
rev abc<|im_end|>
<|im_start|>assistant
cba<|im_end|>
<|im_start|>user
up abc<|im_end|>
<|im_start|>assistant

n tokens: 80 | special ids in stream: [1, 2, 1, 2, 1, 2, 1, 2] ...
round trip ok: True
transformers not installed -> pip install transformers (same idea: a Jinja template stored with the tokenizer)


### 4. Loss masking: train on the assistant tokens only

**What this cell does (plain language):** we turn a chat into `input_ids` and `labels`. Everything the user/system wrote, plus the assistant's role header, gets label `-100`; the assistant's content and its closing `<|im_end|>` keep their real ids (the model must learn to stop). We print a column-by-column table so the alignment is easy to inspect, including a multi-turn example where *both* assistant turns are trained.

In [4]:
def encode_chat_with_mask(messages):
    ids, labels = [], []
    for m in messages:
        header = encode(f"<|im_start|>{m['role']}\n")
        body = encode(m["content"] + "<|im_end|>")
        tail = encode("\n")
        train = m["role"] == "assistant"
        ids += header + body + tail
        labels += [-100] * len(header) + (body if train else [-100] * len(body)) + [-100] * len(tail)
    return ids, labels

demo = [{"role": "user", "content": "rev abc"}, {"role": "assistant", "content": "cba"},
        {"role": "user", "content": "up abc"}, {"role": "assistant", "content": "ABC"}]
ids, labels = encode_chat_with_mask(demo)
show = lambda i: repr(itos[i]) if i >= 4 else itos[i][:6]
print(f"{'pos':>3} {'token':<9} {'label':<9}")
for p, (i, l) in enumerate(zip(ids, labels)):
    print(f"{p:>3} {show(i):<9} {('-100' if l == -100 else show(l)):<9}")
print("trained tokens:", sum(l != -100 for l in labels), "of", len(ids))

pos token     label    
  0 <|im_s    -100     
  1 'u'       -100     
  2 's'       -100     
  3 'e'       -100     
  4 'r'       -100     
  5 '\n'      -100     
  6 'r'       -100     
  7 'e'       -100     
  8 'v'       -100     
  9 ' '       -100     
 10 'a'       -100     
 11 'b'       -100     
 12 'c'       -100     
 13 <|im_e    -100     
 14 '\n'      -100     
 15 <|im_s    -100     
 16 'a'       -100     
 17 's'       -100     
 18 's'       -100     
 19 'i'       -100     
 20 's'       -100     
 21 't'       -100     
 22 'a'       -100     
 23 'n'       -100     
 24 't'       -100     
 25 '\n'      -100     
 26 'c'       'c'      
 27 'b'       'b'      
 28 'a'       'a'      
 29 <|im_e    <|im_e   
 30 '\n'      -100     
 31 <|im_s    -100     
 32 'u'       -100     
 33 's'       -100     
 34 'e'       -100     
 35 'r'       -100     
 36 '\n'      -100     
 37 'u'       -100     
 38 'p'       -100     
 39 ' '       -100     
 40 'a'       -1

### 5. Proof that masked positions contribute nothing

**What this cell does (plain language):** we compute the loss on the example above with a random model in three ways: with labels masked, with labels unmasked (train on everything), and by hand over only the response positions. We also check that the logits at masked positions receive exactly zero gradient, because `ignore_index` removes them from the loss entirely.

In [5]:
import math, torch, torch.utils.checkpoint, torch.nn as nn, torch.nn.functional as F
from dataclasses import dataclass, asdict

@dataclass
class GPTConfig:
    vocab_size: int = 64
    block_size: int = 32
    n_layer: int = 2
    n_head: int = 4
    n_embd: int = 64

class Attn(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.qkv = nn.Linear(c.n_embd, 3 * c.n_embd)
        self.proj = nn.Linear(c.n_embd, c.n_embd)
        self.h = c.n_head
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, self.h, C // self.h).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)  # causal mask built in
        return self.proj(y.transpose(1, 2).reshape(B, T, C))

class Block(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.n_embd), nn.LayerNorm(c.n_embd)
        self.attn = Attn(c)
        self.mlp = nn.Sequential(nn.Linear(c.n_embd, 4 * c.n_embd), nn.GELU(), nn.Linear(4 * c.n_embd, c.n_embd))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.cfg = c
        self.tok = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos = nn.Embedding(c.block_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok.weight          # weight tying
        self.use_ckpt = False
        self.apply(self._init)
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)       # small init => starting loss ~ ln(vocab)
            if getattr(m, 'bias', None) is not None: nn.init.zeros_(m.bias)
    def forward(self, idx, targets=None, ignore_index=-100):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        for b in self.blocks:
            if self.use_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(b, x, use_reentrant=False)
            else:
                x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), ignore_index=ignore_index)
        return logits, loss

V = len(itos)
cfg = GPTConfig(vocab_size=V, block_size=128, n_layer=2, n_head=4, n_embd=64)
torch.manual_seed(0); model = GPT(cfg)
x = torch.tensor([ids[:-1]]); y_masked = torch.tensor([labels[1:]]); y_all = torch.tensor([ids[1:]])   # shift by one

logits, loss_masked = model(x, y_masked)
_, loss_all = model(x, y_all)
sel = y_masked[0] != -100
manual = F.cross_entropy(logits[0][sel], y_masked[0][sel])
print(f"masked loss {loss_masked.item():.4f} | manual (response tokens only) {manual.item():.4f} | unmasked loss {loss_all.item():.4f}")

logits_g = model(x)[0].detach().requires_grad_(True)
F.cross_entropy(logits_g.view(-1, V), y_masked.view(-1), ignore_index=-100).backward()
g = logits_g.grad[0].abs().sum(-1)
print("grad on masked positions (should be 0):", g[~sel].sum().item(), "| on trained positions:", round(g[sel].sum().item(), 4))

masked loss 4.4113 | manual (response tokens only) 4.4113 | unmasked loss 4.2937
grad on masked positions (should be 0): 0.0 | on trained positions: 1.9756


### 6. Collation: dynamic padding and length bucketing

**What this cell does (plain language):** we build a collate function that right-pads `input_ids` with `<pad>`, pads `labels` with `-100`, and builds an attention mask. Then we measure how much of a batch is padding when batches are drawn randomly versus when examples of similar length are grouped (length bucketing). Our examples differ in length only slightly, so we add a few longer multi-turn ones to make the effect visible.

In [6]:
def make_examples(words, tasks=("up", "rev", "dbl")):
    ex = []
    for w in words:
        for t in tasks:
            ex.append(encode_chat_with_mask(chat_record(t, w)["messages"]))
    return ex
train_ex = make_examples(train_words); val_ex = make_examples(val_words)
long_ex = [encode_chat_with_mask(demo)] * 40                              # 2-turn conversations are ~2x longer
pool = train_ex + long_ex
print("examples:", len(pool), "| lengths: min", min(len(i) for i, _ in pool), "max", max(len(i) for i, _ in pool))

def collate(batch):
    L = max(len(i) for i, _ in batch)
    ids_ = torch.full((len(batch), L), PAD); lab = torch.full((len(batch), L), -100); mask = torch.zeros(len(batch), L, dtype=torch.long)
    for k, (i, l) in enumerate(batch):
        ids_[k, :len(i)] = torch.tensor(i); lab[k, :len(l)] = torch.tensor(l); mask[k, :len(i)] = 1
    return ids_, lab, mask

def pad_waste(batches):
    real = sum(len(i) for b in batches for i, _ in b); padded = sum(len(b) * max(len(i) for i, _ in b) for b in batches)
    return 1 - real / padded
rnd = pool[:]; random.Random(0).shuffle(rnd)
random_batches = [rnd[k:k + 16] for k in range(0, len(rnd), 16)]
srt = sorted(pool, key=lambda e: len(e[0]))
bucket_batches = [srt[k:k + 16] for k in range(0, len(srt), 16)]
print(f"padding waste: random batches {100*pad_waste(random_batches):.1f}% | length-bucketed {100*pad_waste(bucket_batches):.1f}%")
b = collate(random_batches[0]); print("batch shapes:", [t.shape for t in b])

examples: 598 | lengths: min 30 max 61
padding waste: random batches 37.1% | length-bucketed 2.0%
batch shapes: [torch.Size([16, 61]), torch.Size([16, 61]), torch.Size([16, 61])]


### 7. Packing: no padding, but you need a block-diagonal mask

**What this cell does (plain language):** we pack many short examples into fixed-length rows separated by `<|endoftext|>` and report the fraction of useful tokens versus padding each example to its own row. Then we show why plain causal attention over a packed row lets a later example 'see' an earlier one (its outputs change), and that a block-diagonal causal mask built from document ids restores the exact standalone result. Position ids must also restart at each example.

In [7]:
def pack(examples, row_len=96):
    stream_ids, stream_lab, doc = [], [], []
    for d, (i, l) in enumerate(examples):
        stream_ids += i + [EOT]; stream_lab += l + [-100]; doc += [d] * (len(i) + 1)   # EOT separates examples
    n = len(stream_ids) // row_len * row_len                                   # drop the ragged tail
    f = lambda s: torch.tensor(s[:n]).view(-1, row_len)
    return f(stream_ids), f(stream_lab), f(doc)
P_ids, P_lab, P_doc = pack(train_ex)
useful = sum(len(i) for i, _ in train_ex)
print(f"packed rows: {tuple(P_ids.shape)} | utilisation packed: ~100% | one-per-row padding to {max(len(i) for i,_ in train_ex)}: {100*useful/(len(train_ex)*max(len(i) for i,_ in train_ex)):.1f}%")

def block_causal_mask(doc_row):
    T = len(doc_row); causal = torch.tril(torch.ones(T, T, dtype=torch.bool))
    return causal & (doc_row[:, None] == doc_row[None, :])                    # same document AND not in the future
def position_ids(doc_row):
    pos = torch.zeros_like(doc_row)
    for d in doc_row.unique(): idx = (doc_row == d).nonzero().squeeze(-1); pos[idx] = torch.arange(len(idx))
    return pos

# Leakage demo with one attention head and no positional info (isolates the mask effect)
torch.manual_seed(0); D = 16; Wq, Wk, Wv = (torch.randn(D, D) for _ in range(3))
E = torch.randn(V, D)
def attend(tok_ids, mask=None):
    h = E[tok_ids]; q, k, v = h @ Wq, h @ Wk, h @ Wv
    return F.scaled_dot_product_attention(q[None], k[None], v[None], attn_mask=mask, is_causal=mask is None)[0]
docA, docB = torch.tensor(train_ex[0][0] + [EOT]), torch.tensor(train_ex[1][0] + [EOT])
packed = torch.cat([docA, docB]); doc_row = torch.cat([torch.zeros(len(docA)), torch.ones(len(docB))]).long()
alone = attend(docB)
leaky = attend(packed)[len(docA):]
masked = attend(packed, block_causal_mask(doc_row))[len(docA):]
print("doc B outputs: plain-causal packing vs standalone max diff:", (leaky - alone).abs().max().item())
print("doc B outputs: block-diagonal mask   vs standalone max diff:", (masked - alone).abs().max().item())
print("position ids restart per example:", position_ids(doc_row)[len(docA) - 2:len(docA) + 3].tolist())

packed rows: (189, 96) | utilisation packed: ~100% | one-per-row padding to 34: 93.1%
doc B outputs: plain-causal packing vs standalone max diff: 16.438461303710938
doc B outputs: block-diagonal mask   vs standalone max diff: 2.86102294921875e-06
position ids restart per example: [29, 30, 0, 1, 2]


### 8. Supervised finetuning: masked loss vs training on everything

**What this cell does (plain language):** we train two identical tiny GPTs from scratch on the chat-formatted data for the same number of steps: one with response-only labels and one with labels on every token. We report exact-match accuracy of greedy generation on held-out words for each task. This is a small-scale experiment, so treat the gap as an illustration of the mechanism, not a general law.

In [8]:
@torch.no_grad()
def generate_reply(model, task, w, add_generation_prompt=True, max_new=10):
    model.eval()
    prompt = apply_chat_template([{"role": "user", "content": f"{task} {w}"}], add_generation_prompt=add_generation_prompt)
    idx = torch.tensor([encode(prompt)])
    for _ in range(max_new):
        nxt = model(idx[:, -cfg.block_size:])[0][:, -1].argmax(-1, keepdim=True)
        if nxt.item() == IM_END: break
        idx = torch.cat([idx, nxt], 1)
    return decode(idx[0, len(encode(prompt)):].tolist())

def accuracy(model, words, **kw):
    return {t: sum(generate_reply(model, t, w, **kw) == fn(w) for w in words) / len(words) for t, fn in TASKS.items()}

def sft(mask_prompt=True, steps=300, bs=32, lr=3e-3, seed=0):
    torch.manual_seed(seed); m = GPT(cfg); opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.0)
    rng = random.Random(seed); m.train()
    for s in range(steps):
        ids_, lab, _ = collate([rng.choice(train_ex) for _ in range(bs)])
        y = (lab if mask_prompt else torch.where(ids_ == PAD, torch.full_like(ids_, -100), ids_))[:, 1:]
        for g in opt.param_groups: g["lr"] = lr * min(1, (s + 1) / 20) * 0.5 * (1 + math.cos(math.pi * s / steps))
        loss = m(ids_[:, :-1], y)[1]; opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(), 1.0); opt.step()
    return m

t0 = time.time()
m_mask, m_all = sft(True), sft(False)
print(f"trained two models in {time.time()-t0:.0f}s")
print("exact match on HELD-OUT words | masked    :", {k: round(v, 2) for k, v in accuracy(m_mask, val_words).items()})
print("exact match on HELD-OUT words | all-tokens:", {k: round(v, 2) for k, v in accuracy(m_all, val_words).items()})
print("exact match on TRAIN words    | masked    :", {k: round(v, 2) for k, v in accuracy(m_mask, train_words[:30]).items()})

trained two models in 11s


exact match on HELD-OUT words | masked    : {'up': 1.0, 'rev': 0.87, 'dbl': 0.7}


exact match on HELD-OUT words | all-tokens: {'up': 0.57, 'rev': 0.0, 'dbl': 0.07}


exact match on TRAIN words    | masked    : {'up': 1.0, 'rev': 0.93, 'dbl': 0.83}


### 9. Template mismatch at inference time

**What this cell does (plain language):** we query the SFT model correctly (with the generation prompt that opens the assistant turn) and incorrectly (omitting it, so the model sees a finished user turn and must guess what comes next). The weights are identical; only the prompt formatting changes. This is the most common reason a freshly finetuned model 'works in training but not in production'.

In [9]:
good = accuracy(m_mask, val_words, add_generation_prompt=True)
bad = accuracy(m_mask, val_words, add_generation_prompt=False)
print("with generation prompt   :", {k: round(v, 2) for k, v in good.items()})
print("without generation prompt:", {k: round(v, 2) for k, v in bad.items()})
print("example (no gen prompt) reply for 'rev abc':", repr(generate_reply(m_mask, "rev", "abc", add_generation_prompt=False)))
print("example (correct)        reply for 'rev abc':", repr(generate_reply(m_mask, "rev", "abc")))

with generation prompt   : {'up': 1.0, 'rev': 0.87, 'dbl': 0.7}
without generation prompt: {'up': 0.0, 'rev': 0.0, 'dbl': 0.0}
example (no gen prompt) reply for 'rev abc': ''
example (correct)        reply for 'rev abc': 'cba'


### 10. Prompt tuning: learning soft-prompt vectors on a frozen model

**What this cell does (plain language):** we freeze the SFT model and train only `K = 4` continuous prompt vectors prepended to the input embeddings, one soft prompt per task. Queries arrive *without* the instruction word (`<|im_start|>user\nabc<|im_end|>...`), and the soft prompt must make the frozen model behave as if it had been told `up`, `rev` or `dbl`. We report exact-match accuracy on held-out words before and after training for each task. Trainable parameters per task: `4 * 64 = 256`. The forward pass is written by hand because the soft prompt enters before the transformer blocks. The teacher-forced loss is a weak progress signal here (once the first response token is given the rest is determined), so we judge by generation accuracy. Expect uneven results: soft prompts steer a frozen model less reliably than finetuning its weights, and in our run the `up` prompt does not get steered at all within this budget (the model has to emit characters that no short vector sequence easily elicits), while `rev` and `dbl` improve.

In [10]:
def forward_with_prompt(model, soft, idx):
    B, T = idx.shape; k = soft.size(1)
    x = torch.cat([soft.expand(B, -1, -1), model.tok(idx)], dim=1)          # [soft prompt ; token embeddings]
    x = x + model.pos(torch.arange(x.size(1)))                              # positions cover prompt + text
    for blk in model.blocks: x = blk(x)
    return model.head(model.ln_f(x))[:, k:]                                 # drop prompt positions: only text positions predict

def example_no_instruction(w, task):
    return encode_chat_with_mask([{"role": "user", "content": w}, {"role": "assistant", "content": TASKS[task](w)}])

frozen = GPT(cfg); frozen.load_state_dict(m_mask.state_dict())
for p in frozen.parameters(): p.requires_grad_(False)
print("frozen params:", sum(p.numel() for p in frozen.parameters()))

@torch.no_grad()
def reply_soft(soft, w, max_new=8):
    idx = torch.tensor([encode(apply_chat_template([{"role": "user", "content": w}], True))]); n0 = idx.size(1)
    for _ in range(max_new):
        nxt = forward_with_prompt(frozen, soft, idx)[:, -1].argmax(-1, keepdim=True)
        if nxt.item() == IM_END: break
        idx = torch.cat([idx, nxt], 1)
    return decode(idx[0, n0:].tolist())
soft_acc = lambda soft, task: sum(reply_soft(soft, w) == TASKS[task](w) for w in val_words) / len(val_words)

def train_soft(task, k=4, steps=300, lr=1e-1, seed=0):
    torch.manual_seed(seed); rng = random.Random(seed)
    soft = nn.Parameter(frozen.tok.weight[torch.randint(4, V, (k,))].detach().clone()[None])   # init from random vocab embeddings
    before = soft_acc(soft, task)
    data = [example_no_instruction(w, task) for w in train_words]
    opt = torch.optim.Adam([soft], lr=lr)
    for s in range(steps):
        ids_, lab, _ = collate([rng.choice(data) for _ in range(32)])
        logits = forward_with_prompt(frozen, soft, ids_[:, :-1])
        loss = F.cross_entropy(logits.reshape(-1, V), lab[:, 1:].reshape(-1), ignore_index=-100)
        opt.zero_grad(); loss.backward(); opt.step()
    return soft, before

souts = {}
for task in TASKS:
    soft, before = train_soft(task); souts[task] = soft
    print(f"task {task:>3}: trainable {soft.numel()} params | exact match before {before:.2f} -> after {soft_acc(soft, task):.2f}")
print("example with the 'dbl' prompt: abc ->", repr(reply_soft(souts["dbl"], "abc")), "| with the 'up' prompt:", repr(reply_soft(souts["up"], "abc")))

frozen params: 113024


task  up: trainable 256 params | exact match before 0.00 -> after 0.00


task rev: trainable 256 params | exact match before 0.80 -> after 0.93


task dbl: trainable 256 params | exact match before 0.00 -> after 0.70
example with the 'dbl' prompt: abc -> 'aabbcc' | with the 'up' prompt: 'ccbccc'


## Common bugs

- **Training and serving templates differ.** A different system prompt, missing `add_generation_prompt`, or an extra space/newline changes token ids and can wreck a finetuned model (section 9). Fix: use one `apply_chat_template` function for both, and unit-test the rendered string.
- **Not masking the prompt (or masking the wrong span).** The model is trained to reproduce user text, and an off-by-one in the mask can hide the first response token or train on the header. Fix: print a token/label table for a few examples (section 4).
- **Forgetting to train on the end-of-turn token.** If `<|im_end|>`/EOS is masked or missing the model never learns to stop and rambles. Fix: keep the end token in the labels and stop generation on it.
- **Masking everything.** A truncated example whose response was cut off has all labels `-100`; the batch loss becomes NaN (0/0) if a whole batch is masked. Fix: drop such examples and check `(labels != -100).any()`.
- **Label shift applied twice (or never).** Passing already-shifted labels to a model that shifts internally (HF causal LMs do) trains the wrong target. Fix: know whether your model shifts; in this notebook we shift manually.
- **Packing without a block-diagonal mask / position-id reset.** Examples leak into each other and later examples see odd positions (section 7). Fix: pass per-document masks (e.g. FlashAttention varlen / `position_ids`) or avoid packing for tiny data.
- **Padding side and pad id in labels.** Using the pad id as a label (instead of `-100`) trains the model to emit padding; left-padding for training with absolute positions shifts positions. Fix: labels `-100` at pads, right-pad for training, left-pad for batched generation.
- **Special tokens split into characters.** If `<|im_start|>` is tokenized as ordinary text it becomes 10+ meaningless tokens and the model cannot learn the structure reliably. Fix: register chat tokens as special tokens and resize embeddings.
- **Soft prompt learning rate too small.** Prompt vectors need a much larger LR than model weights (often 1e-2 to 3e-1) since they start tiny and there are few of them. Fix: sweep the LR separately from model finetuning.

## Exercises

**Exercise 1.** Render the Alpaca record for `dbl abc` without an input field (merge the word into the instruction) and check that it uses the no-input template.

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

In [12]:
# Solution 1
rec = {"instruction": "dbl abc", "input": "", "output": "aabbcc"}
txt = alpaca_prompt(rec)
print(txt); print("uses no-input template:", "### Input:" not in txt)

Below is an instruction. Write a response.

### Instruction:
dbl abc

### Response:

uses no-input template: True


**Exercise 2.** Write a Llama-2-style template (`[INST] user [/INST] assistant </s>`) and render the 2-turn `demo` conversation. List one pro/con compared with ChatML.

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
def llama2_template(messages):
    out = ""
    for m in messages:
        out += f"[INST] {m['content']} [/INST]" if m["role"] == "user" else f" {m['content']} </s>"
    return out
print(llama2_template(demo))
# ChatML marks roles explicitly with dedicated tokens (easy to mask & extend with system/tool roles);
# the [INST] style has no explicit assistant header, so locating the assistant span for masking is fiddlier.

[INST] rev abc [/INST] cba </s>[INST] up abc [/INST] ABC </s>


**Exercise 3.** Mask also the first assistant turn of `demo` and train only on the final reply (`last-turn-only` masking). Show the table of trained positions.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

In [16]:
# Solution 3
ids2, labels2 = encode_chat_with_mask(demo)
last_user_end = len(ids2) - len(encode("<|im_start|>assistant\nABC<|im_end|>\n"))
labels2 = [l if p >= last_user_end else -100 for p, l in enumerate(labels2)]
print("trained:", decode([l for l in labels2 if l != -100]))

trained: ABC<|im_end|>


**Exercise 4.** Compute the packing efficiency as a function of row length (32, 64, 128, 256) for `train_ex` including the dropped ragged tail.

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

In [18]:
# Solution 4
total = sum(len(i) + 1 for i, _ in train_ex)
for L in (32, 64, 128, 256):
    ids_p, _, _ = pack(train_ex, L)
    print(f"row_len {L:>3}: rows {ids_p.shape[0]:>3} | tokens kept {ids_p.numel()}/{total} ({100*ids_p.numel()/total:.1f}%)")

row_len  32: rows 569 | tokens kept 18208/18228 (99.9%)
row_len  64: rows 284 | tokens kept 18176/18228 (99.7%)
row_len 128: rows 142 | tokens kept 18176/18228 (99.7%)
row_len 256: rows  71 | tokens kept 18176/18228 (99.7%)


**Exercise 5.** Train `sft` with a response-only loss for 60, 150 and 300 steps and print accuracy on held-out words. Does accuracy rise smoothly?

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

In [20]:
# Solution 5
for st in (60, 150):
    mm = sft(True, steps=st)
    print(st, "steps:", {k: round(v, 2) for k, v in accuracy(mm, val_words).items()})
print(300, "steps:", {k: round(v, 2) for k, v in accuracy(m_mask, val_words).items()})

60 steps: {'up': 0.0, 'rev': 0.0, 'dbl': 0.0}


150 steps: {'up': 0.9, 'rev': 0.33, 'dbl': 0.5}


300 steps: {'up': 1.0, 'rev': 0.87, 'dbl': 0.7}


**Exercise 6.** Change the soft-prompt length K to 1, 4 and 16 for the `rev` task and compare held-out accuracy (300 steps each). Which K works best here, and is the trend monotonic?

In [21]:
# Your attempt for Exercise 6 here

**Solution 6**

In [22]:
# Solution 6
for k in (1, 4, 16):
    sp, before = train_soft("rev", k=k)
    print(f"K={k:>2}: trainable {sp.numel():>4} | exact match before {before:.2f} -> after {soft_acc(sp, 'rev'):.2f}")
# Results can be non-monotonic on a tiny frozen model; always sweep K and the LR with more than one seed.

K= 1: trainable   64 | exact match before 0.00 -> after 0.00


K= 4: trainable  256 | exact match before 0.80 -> after 0.93


K=16: trainable 1024 | exact match before 0.00 -> after 0.07
